# Análisis de datos históricos de acciones e ingresos

Proyecto final IBM: Tesla (TSLA) y GameStop (GME).

Ejecutar todas las celdas en Colab o Jupyter con conexión a Internet y guardar las salidas antes de la entrega.

In [ ]:
%pip -q install yfinance requests beautifulsoup4 pandas plotly lxml html5lib

In [ ]:
import re
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
from plotly.subplots import make_subplots
import plotly.graph_objects as go
pd.set_option('display.max_columns', None)

## Preparación: funciones para extraer ingresos y graficar

In [ ]:
def extract_quarterly_revenue(url):
    html = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=40)
    html.raise_for_status()
    soup = BeautifulSoup(html.text, 'html.parser')
    for table in soup.find_all('table'):
        if 'quarterly revenue' not in table.get_text(' ', strip=True).lower():
            continue
        rows = []
        for tr in table.find_all('tr'):
            tds = tr.find_all('td')
            if len(tds) < 2:
                continue
            date = tds[0].get_text(strip=True)
            value = tds[1].get_text(strip=True).replace('$', '').replace(',', '')
            if re.fullmatch(r'\d{4}-\d{2}-\d{2}', date) and value:
                rows.append([date, value])
        if rows:
            return pd.DataFrame(rows, columns=['Date', 'Revenue']).drop_duplicates('Date').reset_index(drop=True)
    raise ValueError('No se encontraron ingresos trimestrales en el HTML.')

def make_graph(stock_data, revenue_data, stock):
    stock_data = stock_data.copy()
    revenue_data = revenue_data.copy()
    stock_data['Date'] = pd.to_datetime(stock_data['Date'], utc=True).dt.tz_localize(None)
    revenue_data['Date'] = pd.to_datetime(revenue_data['Date'])
    revenue_data['Revenue'] = pd.to_numeric(revenue_data['Revenue'], errors='coerce')
    end_date = pd.Timestamp('2021-06-30')
    stock_data = stock_data.loc[stock_data['Date'] <= end_date].sort_values('Date')
    revenue_data = revenue_data.loc[revenue_data['Date'] <= end_date].sort_values('Date')
    fig = make_subplots(rows=2, cols=1, subplot_titles=(f'{stock} - Historical Share Price', f'{stock} - Historical Revenue'))
    fig.add_trace(go.Scatter(x=stock_data['Date'], y=stock_data['Close'], mode='lines'), row=1, col=1)
    fig.add_trace(go.Scatter(x=revenue_data['Date'], y=revenue_data['Revenue'], mode='lines'), row=2, col=1)
    fig.update_yaxes(title_text='Precio (USD)', row=1, col=1)
    fig.update_yaxes(title_text='Ingresos (millones USD)', row=2, col=1)
    fig.update_layout(title=f'{stock}: precios e ingresos históricos', height=700, showlegend=False)
    fig.show()
    return fig

## Pregunta 1 — Tesla: precios históricos (`tesla_data.head()`)

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
display(tesla_data.head())

## Pregunta 2 — Tesla: ingresos trimestrales (`tesla_revenue.tail()`)

In [ ]:
tesla_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
tesla_revenue = extract_quarterly_revenue(tesla_url)
display(tesla_revenue.tail())

## Pregunta 3 — GameStop: precios históricos (`gme_data.head()`)

In [ ]:
gamestop = yf.Ticker("GME")
gme_data = gamestop.history(period="max")
gme_data.reset_index(inplace=True)
display(gme_data.head())

## Pregunta 4 — GameStop: ingresos trimestrales (`gme_revenue.tail()`)

In [ ]:
gme_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
gme_revenue = extract_quarterly_revenue(gme_url)
display(gme_revenue.tail())

## Pregunta 5 — Tesla: tablero de precios e ingresos

In [ ]:
tesla_figure = make_graph(tesla_data, tesla_revenue, "Tesla")

## Pregunta 6 — GameStop: tablero de precios e ingresos

In [ ]:
gme_figure = make_graph(gme_data, gme_revenue, "GameStop")

## Entrega

Captura cada una de las seis preguntas con el código y su salida. Ejecuta todo el cuaderno y guarda las salidas; después actualiza el notebook en GitHub, ya que esta versión inicial contiene el código pero no datos descargados ni salidas.